<a href="https://colab.research.google.com/github/samrat-dhakal-11/Google-Colab-Notebooks/blob/main/praticing_adv_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers rank_bm25 openai gradio

In [2]:

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [3]:

import re
import numpy as np

docs = {
    "leave_policy": "Employees receive 20 days of paid annual leave per year. Unused leave can be carried over, up to 5 days, into the next year. Sick leave is separate and gives 10 days per year. Leave requests must be submitted through the HR portal at least two weeks in advance.",
    "remote_work": "Our hybrid policy requires employees to be in the office 3 days per week and allows 2 remote days. Remote days must be approved by your manager in advance. Employees working from home receive a one-time $500 equipment stipend for a home office setup.",
    "expense_policy": "Expenses above $100 require an itemized receipt. Reimbursements are paid within 30 days of submission. Travel must be booked through the approved travel portal. Domestic meals are capped at $60 per day.",
    "security_policy": "Passwords must be changed every 90 days and multi-factor authentication is required on all accounts. Company laptops must have disk encryption enabled. Employees must report suspected phishing emails to security@company.com within one hour.",
}

def chunk_text(text, max_chars=150):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur + [s])) > max_chars:
            chunks.append(" ".join(cur))
            cur = cur[-1:]          # overlap: keep the last sentence
        cur.append(s)
    chunks.append(" ".join(cur))
    return chunks

chunks = [{"doc": name, "text": piece}
          for name, text in docs.items()
          for piece in chunk_text(text)]

print(len(chunks), "chunks")
for c in chunks:
    print(c["doc"], "|", c["text"][:70], "...")

12 chunks
leave_policy | Employees receive 20 days of paid annual leave per year. Unused leave  ...
leave_policy | Unused leave can be carried over, up to 5 days, into the next year. Si ...
leave_policy | Sick leave is separate and gives 10 days per year. Leave requests must ...
remote_work | Our hybrid policy requires employees to be in the office 3 days per we ...
remote_work | Our hybrid policy requires employees to be in the office 3 days per we ...
remote_work | Remote days must be approved by your manager in advance. Employees wor ...
expense_policy | Expenses above $100 require an itemized receipt. Reimbursements are pa ...
expense_policy | Reimbursements are paid within 30 days of submission. Travel must be b ...
expense_policy | Travel must be booked through the approved travel portal. Domestic mea ...
security_policy | Passwords must be changed every 90 days and multi-factor authenticatio ...
security_policy | Passwords must be changed every 90 days and multi-factor authentic

In [4]:

from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
chunk_vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

def dense_rank(q, k=10):
    qv = embedder.encode(q, normalize_embeddings=True)
    return [int(i) for i in np.argsort(-(chunk_vecs @ qv))[:k]]

print(dense_rank("How many days of leave do I get?", k=3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[0, 1, 2]


In [13]:

from rank_bm25 import BM25Okapi

tok = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def bm25_rank(q, k=10):
    return [int(i) for i in np.argsort(-bm25.get_scores(tok(q)))[:k]]

def rrf(rankings, k=60, top=5):
    scores = {}
    for ranking in rankings:
        for rank, i in enumerate(ranking):
            scores[i] = scores.get(i, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)[:top]

def hybrid_rank(q, k=10):
    return rrf([dense_rank(q, 10), bm25_rank(q, 10)], top=k)

In [14]:

test_set = [
    ("How many days of paid annual leave do I get?", "leave_policy"),
    ("Can I carry over unused leave?", "leave_policy"),
    ("How many days a week can I work from home?", "remote_work"),
    ("Do I get money for a home office setup?", "remote_work"),
    ("When do I need a receipt for an expense?", "expense_policy"),
    ("What is the daily meal limit for domestic travel?", "expense_policy"),
    ("How often do I have to change my password?", "security_policy"),
    ("What should I do if I receive a phishing email?", "security_policy"),
]

def hit_rate(retrieve, k):
    hits = sum(any(chunks[i]["doc"] == gold for i in retrieve(q)[:k])
               for q, gold in test_set)
    return hits / len(test_set)

for name, fn in {"dense": dense_rank, "bm25": bm25_rank, "hybrid": hybrid_rank}.items():
    print(f"{name:8s} hit@1={hit_rate(fn, 1):.2f}  hit@3={hit_rate(fn, 3):.2f}")

dense    hit@1=1.00  hit@3=1.00
bm25     hit@1=0.88  hit@3=1.00
hybrid   hit@1=1.00  hit@3=1.00


In [15]:

from openai import OpenAI

client = OpenAI()

def llm(prompt):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.choices[0].message.content

def baseline_answer(q):
    ids = dense_rank(q, k=3)
    ctx = "\n".join(chunks[i]["text"] for i in ids)
    return llm(f"Answer the question using this context.\n\nContext:\n{ctx}\n\nQuestion: {q}")

print(baseline_answer("How many days of paid annual leave do I get?"))

You receive 20 days of paid annual leave per year.
